# lawforge-finetune-etp: QLoRA Goedel-V2-8B on ETP magma proofs

Goedel-V2-8B 4-bit alone fails on SAIR magma problems (out-of-distribution; emits natural-language prose instead of Lean). Teach it the SAIR magma shape by fine-tuning on ETP's 8068 generated proofs filtered to base-tactic-only.

Pipeline:
1. Filter ETP proofs to those using ONLY base Lean tactics (no module helpers).
2. Format as (prompt, response) where prompt is the SAIR-compatible Lean stub and response is the tactic body.
3. QLoRA fine-tune (rank=32, alpha=16, target attn+mlp).
4. Save LoRA adapter to `/kaggle/working/adapter/`.

Smoke knobs: `LAWFORGE_FT_LIMIT` (data cap), `LAWFORGE_FT_STEPS` (training steps).

In [ ]:
!pip -q install --force-reinstall --no-deps \
    'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' \
    --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3' \
    'peft==0.12.0' 'datasets>=2.20' 'trl==0.11.4'

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/lawforge'
if not os.path.isdir(REPO):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/PAMF2/lawforge.git', REPO])
sys.path.insert(0, REPO)
print('repo HEAD:',
      subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
import os, sys, torch
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
print('torch:', torch.__version__, 'arches:', torch.cuda.get_arch_list())
cap = torch.cuda.get_device_capability(0)
sm = cap[0] * 10 + cap[1]
arches = [int(a.replace('sm_', '')) for a in torch.cuda.get_arch_list() if a.startswith('sm_')]
if sm not in arches:
    print(f'ABORT: GPU sm_{sm} not in PyTorch arches {arches}.')
    sys.exit(1)

In [ ]:
import json, re
from pathlib import Path

CORPUS = Path(f'{REPO}/kaggle/finetune_etp/inputs/etp_proofs.jsonl')
LIMIT = int(os.environ.get('LAWFORGE_FT_LIMIT', '0'))  # 0 = all

# A proof is `self-contained` for SAIR if its body uses only base tactics
# and references no ETP-internal lemmas (e.g. RewriteHypothesis.EquationN_implies_M,
# NthRewrites.X, Equation3715_implies_..., Subgraph.Y).
_HELPER = re.compile(
    r'\b(RewriteHypothesis|RewriteGoal|NthRewrites|SimpleRewrites|EquationSearch|'
    r'Subgraph|FromGenerated|Confluence|InstantiateRewrite|Equation\d+_implies_Equation\d+)\b'
)
_ALLOWED_TACTIC = re.compile(
    r'^\s*(intro|intros|have|rw|apply|exact|simp|aesop|calc|refine|symm|cases|'
    r'nth_rewrite|conv|repeat|rfl|decide|assumption|fun|trivial|let|show|by|try|'
    r'·|--|/-|}|{|\)|$|\(|⟨|⟩)'
)


def is_self_contained(proof: str) -> bool:
    if _HELPER.search(proof):
        return False
    # body should start with `by`
    if not proof.strip().startswith('by'):
        return False
    return True


rows = []
kept = 0
with CORPUS.open() as f:
    for line in f:
        row = json.loads(line)
        if is_self_contained(row['lean_proof']):
            rows.append(row)
            kept += 1

print(f'self-contained: {kept}/{kept + (8068 - kept)} ({100*kept/8068:.1f}%)')
if LIMIT > 0:
    rows = rows[:LIMIT]
    print(f'SMOKE cap: {LIMIT}')
print(f'training rows: {len(rows)}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = os.environ.get('LAWFORGE_FT_MODEL', 'Goedel-LM/Goedel-Prover-V2-8B')
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_cfg, device_map='cuda', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=32, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print('mem after model+lora:', torch.cuda.memory_allocated()/1e9, 'GB')

In [ ]:
from datasets import Dataset

LEAN_STMT = (
    'import Mathlib\n'
    'import Aesop\n'
    'set_option maxHeartbeats 400000\n'
    'class Magma (G : Type) where\n'
    '  op : G \u2192 G \u2192 G\n'
    'infixl:70 " \u25c7 " => Magma.op\n\n'
    'theorem sair_implication\n'
    '    (G : Type) [inst : Magma G]\n'
    '    (h : \u2200 x y z w u : G, {eq1})\n'
    '    : \u2200 x y z w u : G, {eq2} := '
)


def to_text(row):
    eq1 = row['eq1_str']
    eq2 = row['eq2_str']
    prompt = (
        'Complete the following Lean 4 code:\n\n```lean4\n'
        + LEAN_STMT.format(eq1=eq1, eq2=eq2)
    )
    proof = row['lean_proof']  # starts with `by`
    full = prompt + proof + '\n```'
    return {'text': full}


ds = Dataset.from_list([to_text(r) for r in rows])
print('dataset:', ds)
print()
print('--- sample text (first 800 chars) ---')
print(ds[0]['text'][:800])
print('...')
print(ds[0]['text'][-400:])

In [ ]:
from trl import SFTTrainer, SFTConfig

STEPS = int(os.environ.get('LAWFORGE_FT_STEPS', '0'))  # 0 = full epoch
EPOCHS = int(os.environ.get('LAWFORGE_FT_EPOCHS', '1'))
BATCH = int(os.environ.get('LAWFORGE_FT_BATCH', '1'))
GRAD_ACC = int(os.environ.get('LAWFORGE_FT_GRADACC', '8'))
LR = float(os.environ.get('LAWFORGE_FT_LR', '2e-4'))
MAX_LEN = int(os.environ.get('LAWFORGE_FT_MAXLEN', '1024'))

OUT = '/kaggle/working/adapter'
cfg = SFTConfig(
    output_dir=OUT,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    max_steps=STEPS if STEPS > 0 else -1,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_LEN,
    packing=False,
)
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=cfg,
    tokenizer=tokenizer,
)
print(f'training: steps={STEPS or "full epoch"} epochs={EPOCHS} batch={BATCH} grad_acc={GRAD_ACC} lr={LR}')
trainer.train()
trainer.save_model(OUT)
print('adapter saved to', OUT)

In [ ]:
# Smoke inference: generate a proof on a held-out problem to verify the
# fine-tune actually produces Lean tactic body (not prose).
import json
from pathlib import Path

DEV = Path(f'{REPO}/kaggle/harvest/inputs/dev_split.jsonl')
p = None
with DEV.open() as f:
    p = json.loads(f.readline())

eq1 = (p.get('equation1') or p.get('hypothesis', '')).replace('*', '\u25c7')
eq2 = (p.get('equation2') or p.get('goal', '')).replace('*', '\u25c7')
prompt = ('Complete the following Lean 4 code:\n\n```lean4\n'
          + LEAN_STMT.format(eq1=eq1, eq2=eq2))

model.eval()
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                         temperature=0.6, top_p=0.95,
                         pad_token_id=tokenizer.pad_token_id)
text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- SMOKE INFERENCE ---')
print('eq1:', eq1)
print('eq2:', eq2)
print()
print('output:')
print(text)